In [ ]:
from pdf2image import convert_from_path, pdfinfo_from_path
import os
import gc
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from pytesseract import pytesseract
from collections import defaultdict
import psutil
import re
import warnings
# import easyocr
import time
import json
import pandas as pd
from tqdm import tqdm
import shutil

# Settings

In [ ]:
batch_size = 5
pdf_folder = r"D:\Programming\Projects\Learning python librairies\OCR"
output_folder = r"D:\Programming\Projects\Learning python librairies\OCR\Outline_Tables\sample"
cropped_folder = r"D:\Programming\Projects\Learning python librairies\OCR\cropped_tables\sample"
poppler_path = r"Propper for image rendering\poppler-26.02.0\Library\bin"
least_horizontal_and_vertical_length = 100
tolerance_of_pixels_in_detecting_multilines = 10
tolerance_of_pixels_in_spliting_multilines = 10
thickness_thresold = 30
tolerance_of_detecting_the_horizontal_and_vertical_line_thickness = 30
error_files_folder = r"D:\Programming\Projects\Learning python librairies\OCR\Outline_Tables\Error Handling"

## Collecting data for training OCR Model
training_data_path = r"E:\VIP Projects\EL ARABY OCR Computer vision Manuals solution\Data Generation\Data collection\Training data for OCR fine tuning"
images_path = os.path.join(training_data_path, "images")
labels_path = os.path.join(training_data_path, "labels")
os.makedirs(images_path, exist_ok=True)
os.makedirs(labels_path, exist_ok=True)
cell_counter = 0

image_quality_dot_per_inch = 500 # please let the image 500 dpi and for more accuracy and don't increase
# render = easyocr.Reader(['en'], gpu=True)

# Functions

In [ ]:
# convert_the_image_to_inverted black and white image
def convert_to_inverted_black_and_white(image):
  gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
  _, bw_image = cv2.threshold(gray_image, 0, 255, cv2.THRESH_BINARY+cv2.THRESH_OTSU)
  if (bw_image == 0).sum() > (bw_image.size / 2):
    sol = bw_image
  else:
    sol = cv2.bitwise_not(bw_image)
  sol = Image.fromarray(sol)
  return sol

In [ ]:
def get_horizontal_and_vertical_lines (img, least_horizontal_and_vertical_length = 50):
  # if img.size == 0:
  #   return -1
  horizontal_kernel_size = max(least_horizontal_and_vertical_length, img.shape[1] // least_horizontal_and_vertical_length)
  horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (horizontal_kernel_size, 1))
  vertical_kernel_size = max(least_horizontal_and_vertical_length, img.shape[0] // least_horizontal_and_vertical_length)
  vertical_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1,vertical_kernel_size))
  
  horizontal_lines = cv2.morphologyEx(img, cv2.MORPH_OPEN, horizontal_kernel)
  vertical_lines = cv2.morphologyEx(img, cv2.MORPH_OPEN, vertical_kernel)
  
  # plt.imshow(cv2.bitwise_or(horizontal_lines, vertical_lines), cmap='gray')
  # plt.show()
  
  return horizontal_lines, vertical_lines

In [ ]:
def check_the_cropped_image_validity_to_table(img, tolerance, thickness_thresold):
  lines = get_horizontal_and_vertical_lines(img, least_horizontal_and_vertical_length=least_horizontal_and_vertical_length)
  if lines == -1:
    print ("This line is ")
    return -1
  horizontal_lines, vertical_lines = lines
  horizontal_projection = np.sum(horizontal_lines > 0, axis=1)
  horizontal_indexes = np.where(horizontal_projection > 0)[0]
  vertical_projection = np.sum(vertical_lines, axis = 0)
  vertical_indexes = np.where(vertical_projection > 0)[0]
  
  horizontal_diff = np.diff(horizontal_indexes)
  vertical_diff = np.diff(vertical_indexes)
  
  split_horizontal_index = np.where(horizontal_diff > tolerance)[0] + 1
  split_vertical_index = np.where(vertical_diff > tolerance)[0] + 1
  
  split_horizontal_index = np.split(horizontal_indexes, split_horizontal_index)
  split_vertical_index = np.split(vertical_indexes, split_vertical_index)
  horizontal_thicknesses = np.array(list(map(lambda x: len(x), split_horizontal_index)))
  vertical_thickness = np.array(list(map(lambda x: len(x), split_vertical_index)))
  if np.any(horizontal_thicknesses > thickness_thresold):
    return -1
  if np.any(vertical_thickness > thickness_thresold):
    return -1
  return None

In [ ]:
def remove_the_outer_border(img): # to work the cell efficently
  horizontal_lines, vertical_lines = get_horizontal_and_vertical_lines(img, least_horizontal_and_vertical_length)
  
  row_projection = np.sum(horizontal_lines > 0, axis=1)
  row_positions = np.where(row_projection > 0)[0]
  
  column_projection = np.sum(vertical_lines > 0, axis=0)
  column_positions = np.where(column_projection > 0)[0]
  img = img.copy()
  img[row_positions,:] = 0
  img[:, column_positions] = 0
  return img

In [ ]:
def the_number_of_rows_and_columns(horizontal_lines, vertical_lines):
  # My problem solving optimization
  row_projection = np.sum(horizontal_lines > 0, axis=1)
  row_positions = np.where(row_projection > 0)[0]
  number_of_rows = len(set(row_positions+range(len(row_positions),0,-1))) - 1

  column_projection = np.sum(vertical_lines > 0, axis=0)
  column_positions = np.where(column_projection > 0)[0]
  number_of_columns = len(set(column_positions+range(len(column_positions),0,-1))) - 1
  return number_of_rows, number_of_columns

In [ ]:
def get_the_horizontal_and_vertical_intersection_points(contours, tolerance=30):
    centroids = []
    for contour in contours:
        sumx = 0
        sumy = 0
        counter = 0
        for position in contour:
            counter += 1
            x, y = position[0]
            sumx += x
            sumy += y
        center_x = sumx // counter
        center_y = sumy // counter
        centroids.append((center_x, center_y))
    # ==========================================================
    # 2. Group Y coordinates -> Horizontal lines
    # ==========================================================
    sorted_y = np.sort(
        np.array([center_y for center_x, center_y in centroids])
    )

    y_splits = np.where(np.diff(sorted_y) > tolerance)[0] + 1

    y_groups = np.split(sorted_y, y_splits)

    # One representative Y for every horizontal line
    horizontals = np.array([
        int(np.median(group))
        for group in y_groups
    ])

    # ==========================================================
    # 3. Group X coordinates -> Vertical lines
    # ==========================================================
    sorted_x = np.sort(
        np.array([center_x for center_x, center_y in centroids])
    )

    x_splits = np.where(np.diff(sorted_x) > tolerance)[0] + 1

    x_groups = np.split(sorted_x, x_splits)

    # One representative X for every vertical line
    verticals = np.array([
        int(np.median(group))
        for group in x_groups
    ])

    # ==========================================================
    # 4. Create mapping:
    # original Y -> representative Y
    # original X -> representative X
    # ==========================================================

    y_mapping = {}

    for group, representative_y in zip(y_groups, horizontals):

        for y in group:
            y_mapping[int(y)] = representative_y

    x_mapping = {}

    for group, representative_x in zip(x_groups, verticals):

        for x in group:
            x_mapping[int(x)] = representative_x

    # ==========================================================
    # 5. Rebuild dictionaries using representative coordinates
    # ==========================================================
    horizontal_fit_lines = defaultdict(list)
    vertical_fit_lines = defaultdict(list)

    for center_x, center_y in centroids:

        representative_y = y_mapping[center_y]
        representative_x = x_mapping[center_x]

        horizontal_fit_lines[representative_y].append(
            center_x
        )

        vertical_fit_lines[representative_x].append(
            center_y
        )

    # ==========================================================
    # 6. Sort X positions inside every horizontal line
    #    and Y positions inside every vertical line
    # ==========================================================
    for y in horizontal_fit_lines:
        horizontal_fit_lines[y].sort()

    for x in vertical_fit_lines:
        vertical_fit_lines[x].sort()

    return (
        horizontals,
        horizontal_fit_lines,
        vertical_fit_lines
    )

In [ ]:
def crop_cell_to_content(img, padding=20):
    row_projection = np.sum(img > 0, axis=1)
    col_projection = np.sum(img > 0, axis=0)

    rows = np.where(row_projection > 0)[0]
    cols = np.where(col_projection > 0)[0]

    if len(rows) == 0 or len(cols) == 0:
        return img

    y1 = max(0, rows[0] - padding)
    y2 = min(img.shape[0], rows[-1] + 1 + padding)

    x1 = max(0, cols[0] - padding)
    x2 = min(img.shape[1], cols[-1] + 1 + padding)

    return img[y1:y2, x1:x2]

In [ ]:
def text_split_lines(img, tolerance = 10, padding = 10):
  row_projection = np.sum(img > 0, axis=1)
  row_indexes = np.where(row_projection > 0)[0]
  gaps = np.diff(row_indexes)
  split_points = np.where(gaps > tolerance)[0] + 1
  lines = np.split(row_indexes, split_points)
  result = []
  for line in lines:
    start = line[0]
    end = line[-1]
    result.append((start-padding, end+padding))
  return result

In [ ]:
def extract_text_from_cells (img, data_train_model = False):
  img = remove_the_outer_border(img)
  # psm = 6 if is_multiline(img) else 7
  config = (
    f"--oem 3 --psm 7 "
    "-c tessedit_char_whitelist="
    " .*()<>[]ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789/\""
    )
  img = crop_cell_to_content(img, 20)
  img = cv2.morphologyEx(img, cv2.MORPH_CLOSE, np.ones((2,2), np.uint8), iterations=2)
  img = cv2.dilate(img, np.ones((2,2), np.uint8), iterations=1)
  gc.collect()
  spliting_lines = text_split_lines(img)
  text = []
  for start, end in spliting_lines:
    padded_image = cv2.copyMakeBorder(img[start:end, :], 20,20,20,20, cv2.BORDER_CONSTANT, value = 0)
    # plt.imshow(padded_image, cmap='gray')
    # plt.show()
    text.append(pytesseract.image_to_string(padded_image, config=config))
    gc.collect()
  text = " ".join(text)
  text = re.sub(r'(?<=\d)["\'°](?=\d)', '*', text)
  text = text.replace("“", '"').replace("”", '"')

  ## For training the OCR model ##
  if data_train_model:
    global cell_counter
    cell_id = f"cell_{cell_counter:06d}"
    image_file = os.path.join(images_path, cell_id + ".png")
    label_file = os.path.join(labels_path, cell_id + ".gt.txt")
    cv2.imwrite(image_file, img)
    with open(label_file, "w", encoding="utf-8") as f:
      f.write(text)
      cell_counter += 1
  ## ---------------------------- ##

  # print("Tesseract text:", text)
  # print ("Tesseract text:"," ".join(" ".join(text.split()).split(" / ")).strip("\""))
  return " ".join(" ".join(text.split()).split(" / ")).strip("\"")

In [ ]:
def table_state(number_of_horizontals, number_of_verticals, contour_nums):
  if number_of_horizontals < 2 or number_of_verticals < 2:
    return -1
  if (number_of_horizontals+1) * (number_of_verticals+1) == contour_nums:
    return True
  if ((number_of_horizontals+1) * (number_of_verticals+1)) - 1 == contour_nums:
    return False
  return -1

In [ ]:
def detect_and_crop_tables(image, output_folder, table_count = 0):
    if image is None:
        raise ValueError("image is None")

    if len(image.shape) != 2:
        raise ValueError("image must be a grayscale/binary image")

    os.makedirs(output_folder, exist_ok=True)
    binary = image
    horizontal_lines, vertical_lines = get_horizontal_and_vertical_lines(binary, least_horizontal_and_vertical_length=least_horizontal_and_vertical_length)
    # print (horizontal_lines)
    table_structure = np.zeros_like(binary)

    cv2.bitwise_or(table_structure, horizontal_lines, dst=table_structure)
    del horizontal_lines
    gc.collect()

    cv2.bitwise_or(table_structure, vertical_lines, dst=table_structure)

    del vertical_lines
    gc.collect()

    connect_kernel = cv2.getStructuringElement(cv2.MORPH_RECT,(5, 5))

    cv2.dilate(table_structure, connect_kernel, dst=table_structure, iterations=2)
    del connect_kernel
    gc.collect()

    contours, _ = cv2.findContours(table_structure, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    del table_structure
    gc.collect()
    boxes = []

    for contour in contours:
        x, y, w, h = cv2.boundingRect(contour)
        area = w * h
        # Ignore very small objects
        if area < 5000:
            continue
        if w < 100 or h < 50:
            continue
        boxes.append((y, x, w, h))
    boxes.sort()
    padding = 10
    for y, x, w, h in boxes:
        x1 = max(0, x - padding)
        y1 = max(0, y - padding)
        x2 = min(binary.shape[1], x + w + padding)
        y2 = min(binary.shape[0], y + h + padding)
        # This is a VIEW, not a full image copy
        table_crop = binary[y1:y2, x1:x2]
        # plt.imshow(table_crop, cmap='gray')
        # plt.show()
        lines = get_horizontal_and_vertical_lines(table_crop, least_horizontal_and_vertical_length=least_horizontal_and_vertical_length)
        if lines == -1:
            continue
        horizontal_lines, vertical_lines = lines
        if check_the_cropped_image_validity_to_table(table_crop, tolerance_of_detecting_the_horizontal_and_vertical_line_thickness, thickness_thresold) == -1:
            # print ("The number of rows and columns")
            # plt.imshow(cv2.bitwise_or(horizontal_lines, vertical_lines), cmap='gray')
            # plt.show()
            # print ("This is not a table")
            continue
        #Problem solving optimization
        number_of_rows, number_of_columns = the_number_of_rows_and_columns(horizontal_lines, vertical_lines)
        

        intersection = cv2.bitwise_and(vertical_lines, horizontal_lines)
        contours, _ = cv2.findContours(intersection, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
        number_of_intersection_points = len(contours)
        if table_state(number_of_rows, number_of_columns, number_of_intersection_points) == -1:
            continue
        table_count += 1
        output_path = os.path.join(output_folder, f"table_{table_count}.png")
        cv2.imwrite(output_path, table_crop)
        print( f"Table {table_count}: " f"x={x1}, y={y1}, " f"width={x2 - x1}, " f"height={y2 - y1}")
    del contours
    del boxes
    gc.collect()

    print("--------------------------------")
    print("Tables detected:", table_count)
    print("Output folder:", output_folder)
    print("--------------------------------")

    return table_count
image = cv2.imread(r"D:\Programming\Projects\Learning python librairies\OCR\Outline_Tables\sample\sample\sample_page_5.png", cv2.IMREAD_GRAYSCALE)
detect_and_crop_tables(image, r"D:\Programming\Projects\Learning python librairies\OCR\cropped_tables")

### testing production 

In [ ]:
# data_folder = r"E:\VIP Projects\EL ARABY OCR Computer vision Manuals solution\Data Generation\Data collection\Images before preprocessing"
# count = 0
# image_number = 0
# for file in os.listdir(data_folder):
#   filepath = os.path.join(data_folder, file)
#   img = cv2.imread(filepath, cv2.IMREAD_GRAYSCALE)
#   count = detect_and_crop_tables(img, r"E:\VIP Projects\EL ARABY OCR Computer vision Manuals solution\Data Generation\Data collection\Test folder",count)
#   image_number += 1
#   print ("==================")
#   print ("Image number",image_number)
#   print ("==================")
#   gc.collect()

### continue the whole program

In [ ]:
def extract_dict_from_image(input_path_image, data_train_model = False): #Must In binary
  img = cv2.imread(input_path_image, cv2.IMREAD_GRAYSCALE)
  # plt.imshow(img, cmap='gray')
  # plt.show()
  lines = get_horizontal_and_vertical_lines(img, least_horizontal_and_vertical_length)
  if lines == -1:
    return -1
  horizontal_lines, vertical_lines = lines
  #Problem solving optimization
  number_of_rows, number_of_columns = the_number_of_rows_and_columns(horizontal_lines, vertical_lines)
  
  intersection = cv2.bitwise_and(vertical_lines, horizontal_lines)
  contours, _ = cv2.findContours(intersection, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
  number_of_intersection_points = len(contours)
  
  # print (f"{number_of_rows}, {number_of_columns}, {number_of_intersection_points}")
  # check header by merged cell
  is_header = False
  if 1 <= ((number_of_rows+1) * (number_of_columns+1)) - number_of_intersection_points:
    is_header = True
    result = {}
    data = {}
  else:
    data = defaultdict(lambda: defaultdict(list))

  if table_state(number_of_rows,number_of_columns,number_of_intersection_points) == -1:
    return -1
  
  horizontals, horizontal_fit_lines, vertical_fit_lines = get_the_horizontal_and_vertical_intersection_points(contours)
  horizontal_fit_lines[horizontals[0]].sort()
  get_header = False
  # print ("Horizontals:", horizontals)
  for idx, _ in enumerate(horizontals): # iterate on each horizontal line from up till before the last
    get_key = False
    if (idx == len(horizontals)-1):
      break
    up = horizontals[idx]
    down = horizontals[idx+1]
    horizontal_fit_lines[up].sort()
    horizontal_fit_lines_up = horizontal_fit_lines[up]
    if is_header: # for header rows that have just only one cell
      TableMode = False
      is_header = False
      get_header = True
      left = horizontal_fit_lines_up[0]
      right = horizontal_fit_lines_up[-1]
      if left >= right:
        continue
      header = extract_text_from_cells(img[up:down, left:right], data_train_model=data_train_model)
      continue

    if not get_header: # for header rows who have more than one cell
      header = defaultdict(list)
      TableMode = True
      get_header = True
      for idx2,vertical_point in enumerate(horizontal_fit_lines_up):
        if idx2 == len(horizontal_fit_lines_up)-1:
          break
        left = horizontal_fit_lines_up[idx2]
        right = horizontal_fit_lines_up[idx2+1]
        if not get_key: # For the key cell in one row
          get_key = True
          title = extract_text_from_cells(img[up:down, left:right], data_train_model=data_train_model)
        else: # For the value of the cell
          header[title].append(extract_text_from_cells(img[up:down, left:right], data_train_model=data_train_model))
      continue

    if TableMode:
      values = defaultdict(dict)
    else:
      values = []

    for idx2, vertical_point in enumerate(horizontal_fit_lines_up):
      # print ("The index",idx)
      if idx2 == len(horizontal_fit_lines_up)-1:
        break
      left = horizontal_fit_lines_up[idx2]
      right = horizontal_fit_lines_up[idx2+1]
      if not get_key: # The first cell in a row
        get_key = True
        key = extract_text_from_cells(img[up:down, left:right], data_train_model=data_train_model)
        continue
      else: # The next cells in a row
        if TableMode:
          data[header[title][idx2-1]][key].append(extract_text_from_cells(img[up:down, left:right], data_train_model=data_train_model))
        else:
          values.append(extract_text_from_cells(img[up:down, left:right], data_train_model=data_train_model))
    if TableMode:
      pass
    else:
      data[key] = values
  if not TableMode:
    result[header] = data
  gc.collect()
  if TableMode:
    return TableMode, data
  return TableMode,result

extract_dict_from_image(r"D:\Programming\Projects\Learning python librairies\OCR\cropped_tables\table_2.png")

In [ ]:
def organising_the_data_for_json(folder_of_cropped_tables, data_train_model=False):
  process = psutil.Process(os.getpid())
  inch_map = {}
  table_inch_content = {}
  result = {}
  for table in os.listdir(folder_of_cropped_tables):
    if not table.lower().endswith(".png"):
      continue
    table = os.path.join(folder_of_cropped_tables, table)
    data = extract_dict_from_image(table, data_train_model=data_train_model)
    if data == -1:
      continue
    table_mode, content = data
    models = content.keys()
    if table_mode:
      table_inch_content = {**table_inch_content, **content}
    else:
      result = {**result, **content}
      for model in models:
        key = re.match(r"\d{2,3}", model)
        if key is None:
          warnings.warn("May cause loss in data")
          print(f"Could not extract inch from model: {model!r}")
          continue
        inch_map[key.group()] = model
  for inch in table_inch_content.keys():
    model = inch_map.get(inch)
    if model is not None:
      result[model] = {**result[inch_map[inch]], **table_inch_content[inch]}

  print (process.memory_info().rss / (1024**2))
  return result
organising_the_data_for_json(r"D:\Programming\Projects\Learning python librairies\OCR\Outline_Tables\sample\sample\Cropped", data_train_model=True)

In [ ]:
def json_to_csv(data, output_path = None, output_file = None):
    rows = []
    for model, specifications in data.items():
        row = {"Model": model}
        for specification, values in specifications.items():
            row[specification] = values[0] if values else ""
        rows.append(row)
    df = pd.DataFrame(rows)
    if output_file is None:
        output_file = os.path.join(output_path, "result.csv")
    df.to_csv(output_file,index=False)
    return df
json_to_csv(organising_the_data_for_json(r"D:\Programming\Projects\Learning python librairies\OCR\Outline_Tables\sample\sample\Cropped"), output_file = "test.csv")

# Code For Running program pipline

In [ ]:
pdf_files = [f for f in os.listdir(pdf_folder) if f.endswith('.pdf')]
errors_count = 0
for pdf_file in tqdm(pdf_files, desc="PDF Processing", leave=True):
  tqdm.write(f"processing.. {pdf_file}")
  try:
    table_count = 0
    pdf_path = os.path.join(pdf_folder, pdf_file)
    pdf_name = os.path.splitext(pdf_file)[0]
    saving_folder = os.path.join(output_folder, pdf_name)
    cropped_saving_folder = os.path.join(saving_folder, "Cropped")
    os.makedirs(saving_folder, exist_ok=True)
    # Get the total number of pages in the PDF
    info = pdfinfo_from_path(pdf_path,poppler_path=poppler_path)
    total_pages = int(info["Pages"])
    print("=" * 10)
    print(f"PDF: {pdf_name}\nTotal pages: {total_pages}\n")
    print("=" * 10)
    for batch_number in range(1, (total_pages // batch_size)+1):
      data = convert_from_path(pdf_path, first_page=batch_number*batch_size-batch_size+1, last_page=batch_number*batch_size, dpi=image_quality_dot_per_inch, poppler_path=poppler_path, thread_count=os.cpu_count())
      for i, page in enumerate(data):
        page = convert_to_inverted_black_and_white(cv2.cvtColor(np.array(page), cv2.COLOR_RGB2BGR))
        page.save(os.path.join(saving_folder, f"{pdf_name}_page_{batch_number*batch_size-batch_size+i+1}.png"), "png")
        table_count = detect_and_crop_tables(np.array(page), cropped_saving_folder, table_count=table_count)
      del data
      gc.collect()

    remaining_pages = total_pages % batch_size
    if remaining_pages > 0:
      data = convert_from_path(pdf_path, first_page=total_pages-remaining_pages+1, last_page=total_pages, dpi=image_quality_dot_per_inch, poppler_path=poppler_path, thread_count=os.cpu_count())
      for i, page in enumerate(data):
        page = convert_to_inverted_black_and_white(cv2.cvtColor(np.array(page), cv2.COLOR_RGB2BGR))
        page.save(os.path.join(saving_folder, f"{pdf_name}_page_{total_pages-remaining_pages+1+i}.png"), "png")
        table_count = detect_and_crop_tables(np.array(page), cropped_saving_folder, table_count=table_count)
      del data
      gc.collect()
    json_file = organising_the_data_for_json(cropped_saving_folder)
    json_path = os.path.join(cropped_saving_folder,"result.json")
    print ("Saving folder:", saving_folder)
    with open(json_path, 'w', encoding='utf-8') as file:
      json.dump(json_file, file, ensure_ascii=False, indent=2)
    with open(json_path, 'r', encoding='utf-8') as file:
      d = json.load(file)
    df = json_to_csv(d, output_path = saving_folder)
    df.head()
    df.to_csv("result.csv", index=False)
  # Extracting data from cropped folder
  except:
    shutil.copy2(pdf_path, error_files_folder)
    errors_count += 1
    print ("="*60)
    print("Error NUmber:",errors_count)
    print ("="*60)